In [4]:
import os
import time
import csv
import subprocess
import psutil
import ast
from pathlib import Path
from dotenv import load_dotenv
import google.generativeai as genai

# ----- CONFIGURACIONES A PROBAR -----
MODELOS = [
    "gemini-2.0-pro-exp-02-05"
]
TEMPERATURAS = [0, 0.5, 1, 2]
# Tres configuraciones para top_p y top_k: determinismo alto, moderado y baja
PARAM_SETS = [
    {"top_p": 0.1, "top_k": 10},   # Más determinista
    {"top_p": 0.5, "top_k": 30},   # Moderado
    {"top_p": 0.95, "top_k": 100}  # Menos determinista
]

# Límite de tiempo para la ejecución de cada script (en segundos)
TIMEOUT_EXEC = 120

# Ruta del archivo CSV con los 15 problemas a probar
INPUT_FILENAME = r"C:\Users\mkoro\Desktop\Transformers\test_dataset\data\processed\leetcode_problems_processed_data - Copy.csv"  # Actualiza la ruta

# Directorio base para guardar resultados (se crearán subdirectorios según configuración)
BASE_OUTPUT_DIR = r"C:\Users\mkoro\Desktop\Transformers\test_dataset\outputs\outputs"
BASE_RESULTS_DIR = r"C:\Users\mkoro\Desktop\Transformers\test_dataset\outputs\results"

# ----- FUNCIONES AUXILIARES -----
def setup_environment():
    """Carga las variables de entorno y configura la API de Gemini."""
    load_dotenv(override=True)
    gemini_api_key = os.environ.get("GEMINI_API_KEY")
    if not gemini_api_key:
        raise EnvironmentError("GEMINI_API_KEY no está definido en las variables de entorno.")
    # Se imprime una versión recortada de la clave
    print(gemini_api_key[:1] + "..." + gemini_api_key[-3:])
    genai.configure(api_key=gemini_api_key)
    return gemini_api_key

def create_chat_session(llm_model: str, temperature: float, top_p: float, top_k: int):
    """Crea y retorna una sesión de chat utilizando el modelo y parámetros especificados."""
    generation_config = {
        "temperature": temperature,
        "top_p": top_p,
        "top_k": top_k,
        "max_output_tokens": 8192,
        "response_mime_type": "text/plain",
    }
    model = genai.GenerativeModel(model_name=llm_model, generation_config=generation_config)
    return model.start_chat(history=[])

def read_problems_from_csv(input_filename: str):
    """Lee el archivo CSV y retorna una lista de problemas (se esperan al menos 15 registros)."""
    problems = []
    with open(input_filename, mode='r', encoding='utf-8') as file:
        csv_reader = csv.reader(file)
        next(csv_reader, None)  # Salta la cabecera si existe
        for fields in csv_reader:
            if len(fields) < 2:
                continue
            problem = {"ID": fields[0], "Description": fields[1]}
            problems.append(problem)
    return problems

def extract_code(response_text: str) -> str:
    """
    Extrae el código Python del texto de respuesta, eliminando bloques markdown.
    Si se encuentra un bloque '```python' o '```', se extrae el contenido interno.
    """
    if "```python" in response_text:
        parts = response_text.split("```python")
        if len(parts) > 1:
            code_content = parts[1].split("```")[0].strip()
            return code_content
    elif "```" in response_text:
        parts = response_text.split("```")
        if len(parts) > 1:
            return parts[1].strip()
    return response_text.strip()

def measure_complexity(code: str) -> str:
    """
    Mide una complejidad algorítmica simple contando el número de bucles y condicionales.
    Esta medida es una aproximación para fines comparativos.
    """
    try:
        tree = ast.parse(code)
    except Exception:
        return "N/A"
    loops = sum(isinstance(node, (ast.For, ast.While)) for node in ast.walk(tree))
    conditionals = sum(isinstance(node, ast.If) for node in ast.walk(tree))
    return f"loops: {loops}, conditionals: {conditionals}"

def run_script_with_metrics(script_path: str, timeout: int = TIMEOUT_EXEC):
    """
    Ejecuta el script Python en 'script_path' con un límite de tiempo y mide:
      - Tiempo de ejecución
      - Memoria máxima utilizada (en MB)
    Devuelve (resultado, tiempo_ejecución, memoria_maxima_MB).
    """
    start_time = time.time()
    process = psutil.Popen(["python", script_path], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    max_memory = 0
    stdout, stderr = b"", b""
    # Monitoreamos el proceso mientras esté activo
    while True:
        if process.poll() is not None:
            stdout, stderr = process.communicate()
            break
        if time.time() - start_time > timeout:
            process.kill()
            return "TIMEOUT", timeout, max_memory / (1024 * 1024)
        try:
            mem = process.memory_info().rss
            if mem > max_memory:
                max_memory = mem
        except psutil.NoSuchProcess:
            break
        time.sleep(0.1)
    execution_time = time.time() - start_time
    result = stdout.decode().strip() if stdout else stderr.decode().strip()
    return result, execution_time, max_memory / (1024 * 1024)

def process_problem(problem: dict, index: int, chat_session, llm_model: str,
                    temperature: float, top_p: float, top_k: int, output_directory: str) -> dict:
    """
    Procesa un problema:
      - Construye el prompt actualizado.
      - Envía la solicitud al modelo.
      - Extrae y limpia el código Python.
      - Guarda el código en un archivo.
      - Ejecuta el script generado, midiendo tiempo, memoria y complejidad.
      - Retorna un diccionario con las métricas y el resultado.
    """
    # Actualizamos el prompt para incluir la ejecución de tests con input/output del problema.
    prompt_text = (
        "Resuelve el siguiente problema en Python. La solución debe implementar una función que, "
        "dado(s) input(s) especificado(s) en el problema, compare su output con el output esperado y "
        "ejecute dichos tests. La función debe imprimir 'True' por cada test que pase y 'False' por cada "
        "test que falle, y finalmente imprimir la cantidad de tests correctos sobre el total. "
        "Proporciona únicamente código Python ejecutable.\n\n"
        f"{problem['Description']}"
    )
    response = chat_session.send_message(prompt_text)
    python_code = extract_code(response.text)
    
    # Guarda el código generado en un archivo
    output_file = Path(output_directory) / f"output_{index + 1}.py"
    with open(output_file, mode='w', encoding='utf-8') as f:
        f.write(python_code)
    print(f"[{llm_model} | T={temperature} | top_p={top_p} | top_k={top_k}] Código guardado en: {output_file}")
    
    # Ejecuta el script generado y obtiene métricas de ejecución
    exec_result, exec_time, mem_usage = run_script_with_metrics(str(output_file), timeout=TIMEOUT_EXEC)
    
    # Calcula una medida simple de complejidad del código
    algo_complexity = measure_complexity(python_code)
    
    # Supongamos que contamos la cantidad de "True" y "False" en el resultado de la ejecución como indicadores de tests pasados/fallados.
    true_count = exec_result.count("True") if isinstance(exec_result, str) else 0
    false_count = exec_result.count("False") if isinstance(exec_result, str) else 0

    # Métricas adicionales (tokens, etc.) se pueden agregar si están disponibles en la respuesta.
    code_metrics = {
        "problem_id": index + 1,
        "code_length": len(python_code),
        "model": llm_model,
        "temperature": temperature,
        "top_p": top_p,
        "top_k": top_k,
    }
    
    return {
        "ID": index + 1,
        "model": llm_model,
        "temperature": temperature,
        "top_p": top_p,
        "top_k": top_k,
        "code": python_code,
        "result": exec_result,
        "true_count": true_count,
        "false_count": false_count,
        "execution_time": round(exec_time, 2),
        "memory_usage_MB": round(mem_usage, 2),
        "algorithmic_complexity": algo_complexity
    }
def process_problem_with_retry(problem: dict, index: int, chat_session, model: str,
                                 temp: float, top_p: float, top_k: int, output_directory: str,
                                 max_retries: int = 3) -> dict:
    """
    Intenta procesar el problema, reintentando en caso de error 429.
    """
    retries = 0
    while retries < max_retries:
        try:
            result = process_problem(problem, index, chat_session, model, temp, top_p, top_k, output_directory)
            return result
        except Exception as e:
            error_msg = str(e)
            if "429" in error_msg:
                retries += 1
                print(f"Error 429 en el problema {index+1} (modelo: {model}, Temp: {temp}, top_p: {top_p}, top_k: {top_k}). Reintentando ({retries}/{max_retries})...")
                # Espera un tiempo antes de reintentar (por ejemplo, 30 segundos multiplicado por el número de reintentos)
                time.sleep(30 * retries)
            else:
                # Si es otro error, lo lanzamos de nuevo
                raise
    # Si se exceden los reintentos, se lanza la excepción
    raise Exception(f"Problema {index+1} falló tras {max_retries} reintentos por error 429.")

def main():
    try:
        setup_environment()
        problems = read_problems_from_csv(INPUT_FILENAME)
        if not problems:
            print("No se encontraron problemas en el archivo CSV.")
            return

        max_problems = 5  # Se procesarán 5 problemas
        results = []
        
        # Itera sobre cada modelo, temperatura y configuración de top_p/top_k
        for model in MODELOS:
            for temp in TEMPERATURAS:
                for params in PARAM_SETS:
                    top_p = params["top_p"]
                    top_k = params["top_k"]
                    print(f"Procesando: {model} | Temp: {temp} | top_p: {top_p} | top_k: {top_k}")
                    
                    # Crear directorios específicos para esta configuración
                    output_directory = Path(BASE_OUTPUT_DIR) / f"temperature-{temp}" / model / f"top_p-{top_p}_top_k-{top_k}"
                    results_directory = Path(BASE_RESULTS_DIR) / f"temperature-{temp}" / model / f"top_p-{top_p}_top_k-{top_k}"
                    output_directory.mkdir(parents=True, exist_ok=True)
                    results_directory.mkdir(parents=True, exist_ok=True)
                    
                    # Inicializa la sesión de chat para esta configuración
                    chat_session = create_chat_session(model, temp, top_p, top_k)
                    
                    for i in range(min(len(problems), max_problems)):
                        output_file = output_directory / f"output_{i + 1}.py"
                        # Si el archivo ya existe, se asume que este problema ya fue procesado
                        if output_file.exists():
                            print(f"[{model} | T={temp} | top_p={top_p} | top_k={top_k}] Problema {i + 1} ya procesado, omitiendo.")
                            continue

                        print(f"Procesando Problema {i + 1} de {max_problems}")
                        start_time = time.time()
                        try:
                            # Se utiliza la función con reintentos
                            result = process_problem_with_retry(problems[i], i, chat_session, model, temp, top_p, top_k, str(output_directory))
                            results.append(result)
                        except Exception as e:
                            print(f"Error en el problema {i + 1}: {e}")
                            results.append({
                                "ID": i + 1,
                                "model": model,
                                "temperature": temp,
                                "top_p": top_p,
                                "top_k": top_k,
                                "code": "",
                                "result": str(e),
                                "true_count": "ERROR",
                                "false_count": "ERROR",
                                "execution_time": "ERROR",
                                "memory_usage_MB": "ERROR",
                                "algorithmic_complexity": "ERROR"
                            })
                        elapsed = time.time() - start_time
                        # Asegura una pausa mínima para respetar límites de tasa (15 segundos por iteración)
                        if i < min(len(problems), max_problems) - 1:
                            sleep_time = max(0, 15 - elapsed)
                            time.sleep(sleep_time)
        
        # Guarda todos los resultados en un archivo CSV (se agrupan todas las configuraciones)
        results_csv = Path(BASE_RESULTS_DIR) / "results_completo.csv"
        with open(results_csv, mode='w', newline='', encoding='utf-8') as csvfile:
            fieldnames = [
                "ID", "model", "temperature", "top_p", "top_k",
                "code", "result", "true_count", "false_count",
                "execution_time", "memory_usage_MB", "algorithmic_complexity"
            ]
            writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(results)
        print(f"CSV de resultados guardado en: {results_csv}")
    
    except Exception as ex:
        print(f"Ocurrió un error: {ex}")

if __name__ == "__main__":
    main()



A...VPc
Procesando: gemini-2.0-pro-exp-02-05 | Temp: 0 | top_p: 0.1 | top_k: 10
[gemini-2.0-pro-exp-02-05 | T=0 | top_p=0.1 | top_k=10] Problema 1 ya procesado, omitiendo.
[gemini-2.0-pro-exp-02-05 | T=0 | top_p=0.1 | top_k=10] Problema 2 ya procesado, omitiendo.
[gemini-2.0-pro-exp-02-05 | T=0 | top_p=0.1 | top_k=10] Problema 3 ya procesado, omitiendo.
[gemini-2.0-pro-exp-02-05 | T=0 | top_p=0.1 | top_k=10] Problema 4 ya procesado, omitiendo.
[gemini-2.0-pro-exp-02-05 | T=0 | top_p=0.1 | top_k=10] Problema 5 ya procesado, omitiendo.
Procesando: gemini-2.0-pro-exp-02-05 | Temp: 0 | top_p: 0.5 | top_k: 30
[gemini-2.0-pro-exp-02-05 | T=0 | top_p=0.5 | top_k=30] Problema 1 ya procesado, omitiendo.
[gemini-2.0-pro-exp-02-05 | T=0 | top_p=0.5 | top_k=30] Problema 2 ya procesado, omitiendo.
[gemini-2.0-pro-exp-02-05 | T=0 | top_p=0.5 | top_k=30] Problema 3 ya procesado, omitiendo.
[gemini-2.0-pro-exp-02-05 | T=0 | top_p=0.5 | top_k=30] Problema 4 ya procesado, omitiendo.
[gemini-2.0-pro-exp-